## Asistente experto en conocimiento

### Un agente de respuesta a preguntas que actúe como asistente experto en conocimiento
### Destinado a los empleados de Insurellm, una empresa de tecnología aplicada a los seguros
### El agente debe ser preciso y la solución debe ser económica.

Este proyecto utilizará RAG (Retrieval Augmented Generation) para garantizar que nuestro asistente de preguntas y respuestas tenga una alta precisión.

## HOY:

- Parte A: Dividiremos nuestros documentos en FRAGMENTOS
- Parte B: Codificaremos nuestros FRAGMENTOS en VECTORES y los introduciremos en Chroma
- Parte C: Visualizaremos nuestros vectores

Traducción realizada con la versión gratuita del traductor DeepL.com

### PARTE A: Dividir nuestros documentos en partes

In [55]:
import os
import glob
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from transformers import AutoTokenizer

In [56]:
# price is a factor for our company, so we're going to use a low cost models
db_name = "test_db"
load_dotenv(override=True)

MODEL = "NousResearch/Meta-Llama-3-8B"
openai = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)


In [57]:
# ¿Cuántos caracteres hay en el documento?

knowledge_base_path = "../knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 15 files in the knowledge base
Total characters in knowledge base: 74,408


In [58]:
# Cuantos tokens hay en el documento?
tokenizer = AutoTokenizer.from_pretrained(MODEL)
encoding = tokenizer.encode(entire_knowledge_base)
tokens = tokenizer.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for NousResearch/Meta-Llama-3-8B: 19,961


In [59]:
# Carga todo el contenido de la base de conocimientos utilizando los cargadores de LangChain

folders = glob.glob("../knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 15 documents


In [60]:
documents[1]

Document(metadata={'source': '..\\knowledge-base\\compañia\\cultura.md', 'doc_type': 'compañia'}, page_content='# Cultura de AgroLLM\n\n## Declaración de Visión\n\nRevolucionar el sector agropecuario mediante tecnología innovadora que haga el conocimiento técnico, el control fitosanitario y la gestión de ayudas accesible, transparente y ágil para cada agricultor, ganadero y técnico agrícola.\n\n## Declaración de Misión\n\nEmpoderar a los productores del sector primario, cooperativas y oficinas de extensión agraria con soluciones de software de vanguardia que simplifiquen la burocracia, optimicen los recursos hídricos en zonas áridas y mejoren la toma de decisiones en el campo. Combinando el conocimiento técnico tradicional con la innovación de la Inteligencia Artificial, estamos construyendo el futuro del sector primario.\n\n## Valores Fundamentales\n\n### Innovación en el Campo (Innovation First)\n\nDesafiamos los métodos tradicionales de consulta de información y apostamos por la res

In [61]:
# Dividir en chunks usando el RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 113 chunks
First chunk:

page_content='# Empleo en AgroLLM

## ¿Por qué unirte a AgroLLM?

En AgroLLM no solo desarrollamos software: estamos revolucionando el sector agropecuario. Desde nuestra fundación en 2015, hemos evolucionado de ser una startup de rápido crecimiento a una empresa altamente rentable y eficiente. Contamos con un equipo de 32 profesionales excepcionales que gestionan 32 contratos activos con las principales cooperativas, asociaciones de productores y comunidades de regantes a través de nuestras ocho líneas de producto basadas en Inteligencia Artificial.' metadata={'source': '..\\knowledge-base\\compañia\\carreras.md', 'doc_type': 'compañia'}


In [62]:
chunks[100]

Document(metadata={'source': '..\\knowledge-base\\productos\\Preciollm.md', 'doc_type': 'productos'}, page_content='- **Q2 2025:** lanzamiento de PrecioLLM versión 1.0 con el motor RAG de texto básico para la consulta de boletines de precios semanales e ingesta manual de fichas de costes de producción.\n- **Q4 2025:** incorporación del modelo de visión multimodal (Llama 3.2 Vision) para la extracción automatizada de tablas de precios manuscritas o escaneadas desde tablones de anuncios físicos de las cooperativas.\n- **Q2 2026:** integración nativa mediante herramientas de LangChain con sensores y software de gestión de almacén (ERP) para correlacionar picos de precios con la oferta real almacenada en tiempo real.\n- **Q4 2026:** lanzamiento del módulo de seguros agrarios paramétricos en colaboración con Agroseguro, evaluando automáticamente la pérdida de rendimiento económico por inclemencias meteorológicas severas.')

### PART B: Construir vectores y almacenarnos en un chroma

In [63]:


embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore creada con {vectorstore._collection.count()} documentos")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vectorstore creada con 113 documentos


In [64]:
# Vamos a investigar los vectores
collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 113 vectors with 384 dimensions in the vector store


### Visualizarlos

In [65]:
# Prework
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['productos', 'empleados', 'contratos', 'compañia'].index(t)] for t in doc_types]

In [66]:
# ¡A los humanos nos resulta más fácil visualizar las cosas en 2D!
# Reduce la dimensionalidad de los vectores a 2D utilizando t-SNE
# (incrustación estocástica de vecinos con distribución t)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [38]:
# Representacion en 3D
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()